# Grok-cv-keypoint-detection

Computer vision smoke demo: **keypoint-detection** on Kaggle **GPU T4 x2**.

Writes `/kaggle/working/result.json` and prints `SMOKE_OK` on success.


In [ ]:
import json, os, sys, time, traceback, math, gc
from pathlib import Path

import torch
import numpy as np

TASK = os.environ.get("GROK_TASK", "keypoint-detection")
NOTEBOOK = "Grok-cv-keypoint-detection"
OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

def log(*a):
    print(*a, flush=True)

def gpu_info():
    info = {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
    }
    log("GPU:", info)
    assert info["cuda_available"], "CUDA required — enable Kaggle GPU T4x2"
    assert info["device_count"] >= 1
    return info

def device0():
    return torch.device("cuda:0")

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_result(payload: dict):
    payload = {
        "ok": True,
        "notebook": NOTEBOOK,
        "task": TASK,
        "domain": "cv",
        **payload,
    }
    path = OUT / "result.json"
    path.write_text(json.dumps(payload, indent=2, default=str))
    log("wrote", path)
    log(json.dumps(payload, indent=2, default=str)[:2000])
    log("SMOKE_OK")
    return payload

def load_sample_image(size=(384, 384)):
    """Download a small sample RGB image (internet on)."""
    from PIL import Image
    import urllib.request
    urls = [
        "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=640",  # cat
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
        "https://picsum.photos/seed/grokcv/512/512",
    ]
    last = None
    for u in urls:
        try:
            fn = OUT / "sample.jpg"
            urllib.request.urlretrieve(u, fn)
            img = Image.open(fn).convert("RGB")
            img = img.resize(size)
            log("sample image", u, img.size)
            return img
        except Exception as e:
            last = e
            log("sample fetch fail", u, e)
    # synthetic fallback
    from PIL import ImageDraw
    img = Image.new("RGB", size, (30, 30, 40))
    d = ImageDraw.Draw(img)
    d.rectangle([40, 40, size[0]-40, size[1]-40], outline=(0, 200, 255), width=6)
    d.ellipse([size[0]//3, size[1]//3, 2*size[0]//3, 2*size[1]//3], fill=(255, 120, 40))
    log("using synthetic sample", last)
    return img

t_start = time.time()
info = gpu_info()


In [ ]:
try:
    # Keypoint detection — Keypoint R-CNN (COCO person keypoints)
    import torchvision
    from torchvision.transforms import functional as F
    from PIL import Image, ImageDraw

    img = load_sample_image((320, 320))
    # Draw a crude stick-person so detector has a chance on synthetic too
    draw = ImageDraw.Draw(img)
    draw.ellipse([140, 40, 180, 80], outline=(255, 220, 180), width=3)
    draw.line([160, 80, 160, 180], fill=(0, 200, 255), width=4)
    draw.line([160, 110, 120, 140], fill=(0, 200, 255), width=4)
    draw.line([160, 110, 200, 140], fill=(0, 200, 255), width=4)
    draw.line([160, 180, 130, 250], fill=(0, 200, 255), width=4)
    draw.line([160, 180, 190, 250], fill=(0, 200, 255), width=4)

    weights = torchvision.models.detection.KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(weights=weights).to(device0()).eval()
    x = F.to_tensor(img).to(device0())
    t0 = time.time()
    with torch.inference_mode():
        pred = model([x])[0]
    dt = time.time() - t0
    scores = pred["scores"].detach().cpu()
    keep = scores >= 0.5
    kps = pred["keypoints"][keep].cpu().numpy() if keep.any() else np.zeros((0, 17, 3))
    sc = scores[keep].tolist()
    log("persons", len(sc), "keypoints_shape", kps.shape)
    # save first skeleton overlay
    vis = img.copy()
    d = ImageDraw.Draw(vis)
    if len(kps):
        for x,y,v in kps[0]:
            if v > 0:
                d.ellipse([x-3, y-3, x+3, y+3], fill=(255, 0, 0))
    vis.save(OUT / "keypoints.png")
    clear_mem()
    save_result({
        "model": "keypointrcnn_resnet50_fpn",
        "num_instances": int(len(sc)),
        "scores": [float(s) for s in sc[:5]],
        "keypoints_shape": list(kps.shape),
        "inference_s": dt,
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    })
except Exception as e:
    log("TASK_FAILED", type(e).__name__, e)
    traceback.print_exc()
    err = {
        "ok": False,
        "notebook": NOTEBOOK,
        "task": TASK,
        "error": repr(e),
        "elapsed_s": time.time() - t_start,
        "gpu": info,
    }
    (OUT / "result.json").write_text(json.dumps(err, indent=2, default=str))
    raise
